# Week 2 — Day 3-4: BLIP fine-tuning experiments (run on Kaggle GPU)

**This notebook runs on Kaggle, not locally** — the project's `.venv` is CPU-only by design; all
fine-tuning happens on Kaggle's free GPU.

## Setup (do this before running)
1. Create a new Kaggle Notebook and upload/paste this file.
2. **Settings → Accelerator → GPU** (T4 x2 or P100).
3. **Add Data** → search "Flickr8k" → add the `adityajn105/flickr8k` dataset (or whichever
   Flickr8k dataset you use — it should contain `Images/` and `captions.txt`). If the input path
   differs from `/kaggle/input/flickr8k`, update `KAGGLE_INPUT_DIR` in the first code cell.
4. Run all cells. Total runtime: roughly 20-40 minutes for all 3 configs on a T4.
5. When done, download `/kaggle/working/week2_finetune_results.json` and
   `/kaggle/working/week2_finetune_results.csv` from the notebook's **Output** tab and drop them
   into this repo's `data/processed/` folder — the local `week2_day4_finetune_comparison.ipynb`
   notebook picks them up from there to log everything into MLflow and pick a winner.
   (Model checkpoints are also saved per-config if you want to download and use one directly;
   they're large, so only grab the one you need.)

## Approach

Per the Week 2 plan: **freeze the vision encoder, fine-tune only the text decoder** — cheaper and
less prone to overfitting on a small subset than full fine-tuning. Three configs isolate different
axes so we can tell what actually moved the needle:

| Config | Learning rate | Decoding at eval | Purpose |
|---|---|---|---|
| A — baseline fine-tune | 5e-5 | greedy | default fine-tuning config |
| B — lower LR | 1e-5 | greedy | check if a gentler LR improves stability/quality |
| C — beam search | 5e-5 (same weights as A) | beam search (num_beams=4) | isolate decoding-strategy effect, no retraining needed |


### 1. Setup

In [ ]:
import os
import json
import time
import random
import pandas as pd
import torch
import evaluate
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration

KAGGLE_INPUT_DIR = "/kaggle/input/flickr8k"   # adjust if your dataset slug differs
OUTPUT_DIR = "/kaggle/working"

CAPTIONS_PATH = os.path.join(KAGGLE_INPUT_DIR, "captions.txt")
IMG_DIR = os.path.join(KAGGLE_INPUT_DIR, "Images")

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "No GPU detected -- check Settings > Accelerator on Kaggle."
print(f"Using device: {device}")

MODEL_NAME = "Salesforce/blip-image-captioning-base"

df = pd.read_csv(CAPTIONS_PATH)
df.columns = ["image", "caption"]
print(f"Total captions: {len(df)}, unique images: {df['image'].nunique()}")

### 2. Train / eval split

3,000 training pairs, 100 held-out eval images (disjoint from training) — enough to see a
directional signal within a Kaggle session without every config taking forever.

In [ ]:
random.seed(42)
all_images = df["image"].drop_duplicates().tolist()
random.shuffle(all_images)

eval_images = all_images[:100]
train_images = all_images[100:100 + 1500]   # ~1500 images x ~2 captions each ~= 3000 training pairs

train_df = df[df["image"].isin(train_images)].groupby("image").head(2).reset_index(drop=True)
eval_refs = [df[df["image"] == img]["caption"].tolist() for img in eval_images]

print(f"Training pairs: {len(train_df)} (from {len(train_images)} images)")
print(f"Eval images: {len(eval_images)}")

### 3. Dataset / collator

In [ ]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)

class FlickrFineTuneDataset(Dataset):
    def __init__(self, dataframe, img_dir):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        return image, row["caption"]

def collate_fn(batch):
    images, captions = zip(*batch)
    inputs = processor(
        images=list(images), text=list(captions),
        padding="max_length", truncation=True, max_length=32, return_tensors="pt",
    )
    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels
    return inputs

train_dataset = FlickrFineTuneDataset(train_df, IMG_DIR)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
print(f"{len(train_loader)} batches/epoch")

### 4. Train + eval functions

In [ ]:
def train_model(lr, num_epochs=1):
    model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
    for p in model.vision_model.parameters():
        p.requires_grad = False   # freeze vision encoder

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr)

    model.train()
    losses = []
    for epoch in range(num_epochs):
        for batch in tqdm(train_loader, desc=f"train lr={lr} epoch={epoch}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            losses.append(loss.item())
    return model, losses


@torch.no_grad()
def evaluate_model(model, decoding="greedy"):
    model.eval()
    gen_kwargs = {"max_new_tokens": 30}
    if decoding == "beam":
        gen_kwargs["num_beams"] = 4
    else:
        gen_kwargs["num_beams"] = 1

    predictions = []
    for img_name in tqdm(eval_images, desc=f"eval decoding={decoding}"):
        image = Image.open(os.path.join(IMG_DIR, img_name)).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        out = model.generate(**inputs, **gen_kwargs)
        predictions.append(processor.decode(out[0], skip_special_tokens=True))

    bleu = evaluate.load("sacrebleu").compute(predictions=predictions, references=eval_refs)
    rouge = evaluate.load("rouge").compute(predictions=predictions, references=eval_refs)
    return {
        "bleu": bleu["score"],
        "rouge1": rouge["rouge1"] * 100,
        "rouge2": rouge["rouge2"] * 100,
        "rougeL": rouge["rougeL"] * 100,
    }, predictions

### 5. Run the 3 configs

In [ ]:
results = {}

# Config A: baseline fine-tune, greedy decoding
t0 = time.time()
model_a, losses_a = train_model(lr=5e-5)
metrics_a, preds_a = evaluate_model(model_a, decoding="greedy")
results["A_lr5e-5_greedy"] = {
    "lr": 5e-5, "decoding": "greedy", "final_train_loss": losses_a[-1],
    "train_time_s": time.time() - t0, **metrics_a,
}
model_a.save_pretrained(os.path.join(OUTPUT_DIR, "config_A_checkpoint"))

# Config B: lower learning rate, greedy decoding
t0 = time.time()
model_b, losses_b = train_model(lr=1e-5)
metrics_b, preds_b = evaluate_model(model_b, decoding="greedy")
results["B_lr1e-5_greedy"] = {
    "lr": 1e-5, "decoding": "greedy", "final_train_loss": losses_b[-1],
    "train_time_s": time.time() - t0, **metrics_b,
}
model_b.save_pretrained(os.path.join(OUTPUT_DIR, "config_B_checkpoint"))

# Config C: same weights as config A, beam search decoding instead of greedy
metrics_c, preds_c = evaluate_model(model_a, decoding="beam")
results["C_lr5e-5_beam"] = {
    "lr": 5e-5, "decoding": "beam", "final_train_loss": losses_a[-1],
    "train_time_s": results["A_lr5e-5_greedy"]["train_time_s"], **metrics_c,
}

results_df = pd.DataFrame(results).T
results_df

### 6. Save results for download

In [ ]:
with open(os.path.join(OUTPUT_DIR, "week2_finetune_results.json"), "w") as f:
    json.dump(results, f, indent=2)

results_df.to_csv(os.path.join(OUTPUT_DIR, "week2_finetune_results.csv"))

best_config = results_df["bleu"].astype(float).idxmax()
print(f"Best config by BLEU: {best_config}")
print("\nDownload week2_finetune_results.json/csv from the Output tab and copy them into")
print("this repo's data/processed/ folder, then run notebooks/week2_day4_finetune_comparison.ipynb locally.")